# Xử lý dữ liệu VietNews — v2

Pipeline xử lý dữ liệu **read-only đối với raw**, đánh dấu thay vì xóa, kết quả ghi ra `data/processed/v2/`.

## Quy tắc chung
1. Mọi trường hợp nghi ngờ → `review_queue/`, **dừng để nhóm quyết định, không tự xóa hàng**.
2. `title` chỉ là metadata; cặp dùng là **`article` → `abstract`**.
3. Không ghi đè v1; mỗi lần chạy lại xuất sang version mới (v2, v3, ...).
4. **Không cắt `article` xuống 1024 token trong file lưu trữ**; việc cắt chỉ xảy ra lúc tokenize để train.
5. Chia **split_v2 mới 65/15/20** (seed 42, stratify độ dài article+abstract, nhóm trùng đi cùng nhau);
   giữ `original_split` để truy vết; **test mới bị khóa, không dùng để chọn tham số**.
6. Đọc raw bằng parquet, không sửa file gốc; ghi checksum raw.

## Phần 1. Mục tiêu & cấu hình

Cấu hình tập trung: version dataset, seed, đường dẫn raw/output, ngưỡng kiểm tra.
Khi chạy lại chỉ cần đổi `VERSION` ở cell này.

In [1]:
"""1. Cấu hình tập trung - chạy lại được."""
from __future__ import annotations

import hashlib
import json
import os
import re
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

# ------------------------------------------------------------ tham số chạy
VERSION = "v2"                     # đổi khi chạy lại: v3, v4, ...
SEED = 42                          # dùng chung cho mọi sampling
NOW_UTC = datetime.now(timezone.utc).isoformat(timespec="seconds")

IS_SMOKE = os.environ.get("XULY_V2_SMOKE") == "1"

ROOT = Path.cwd()
DATA_RAW = ROOT / "data" / "raw" / "vietnews"
OUT_DIR = ROOT / "data" / "processed" / VERSION
OUT_DIR_SMOKE = ROOT / "data" / "processed" / f"{VERSION}_smoke"
OUT_DIR = OUT_DIR_SMOKE if IS_SMOKE else OUT_DIR
REPORTS_DIR = ROOT / "reports"
REVIEW_DIR = OUT_DIR / "review_queue"
REVIEW_APPROVAL = ROOT / "data" / "review" / VERSION / "approval.json"
for d in (OUT_DIR, REPORTS_DIR, REVIEW_DIR, REVIEW_APPROVAL.parent):
    d.mkdir(parents=True, exist_ok=True)

EXPECT_EVAL_PACK = 300   # số ID tối thiểu của eval pack (đủ khi chạy full)

# ------------------------------------------------------------ replay v1 (đã duyệt)
V1_REPLAY_FILE = ROOT / "data" / "processed" / "v1" / "vietnews_merged.parquet"
# Danh sách (split, guid) v1 đã bỏ ở các bước đã duyệt:
# pii_fix (3) + invalid_pair_fix (1) + cuối cùng (1). Pipeline xác minh bằng diff
# với v1: chỉ giữ row có trong v1, không đoán lý do.

# ------------------------------------------------------------ cổng duyệt review
APPROVAL_FILE = ROOT / "data" / "review" / VERSION / "approval.json"
REVIEW_POLICY = {
    "pii": "mask",          # quyết định đã chốt: mask email/URL/điện thoại
    "fact": "keep",         # cờ fact-check: giữ kèm ghi chú trong data card
    "audit": "keep",
    "label": "keep",
    "exact_dup": "keep",    # v1 giữ bản trùng nhưng dồn cùng split_v2
    "near_dup_xsplit": "keep",
    "v1_replay": "remove",  # 5 dòng v1 đã xoá
}

# ------------------------------------------------------------ ngưỡng kiểm tra
THRESHOLDS = {
    "flag_empty": True,               # rỗng -> lỗi
    "min_abstract_chars": 20,         # abstract quá ngắn
    "max_abstract_to_article": 0.75,  # abstract >= 75% article (bài dài) -> cờ
    "near_dup_jaccard": 0.90,         # 2 văn bản "gần trùng" (ngưỡng như §6 sàng lọc)
    "max_article_chars": 20000,       # article quá dài (cảnh báo, không cắt)
    "max_abstract_chars": 2000,
    "near_dup_cap_group": 50,         # giới hạn nhóm xét near-dup (xấp xỉ)
    "cut_512": 512,                   # ngưỡng cắt ngắn
    "cut_1024": 1024,                 # ngưỡng cắt dài
    "cut_cover_min": 0.9,             # cần >= 90% thực tố nằm trong phần giữ lại
}

TOKEN_BUCKETS = {"short": (0, 512), "mid": (512, 1024), "long": (1024, None)}
VIT5_TOKENIZER_DIR = ROOT / "models" / "vit5-base"

# -----------------------------------------------------------------------
# Phần 10: chia split_v2 - tỷ lệ mục tiêu, stratify theo độ dài article+abstract
SPLIT_V2_ORDER = ["train", "validation", "test"]
SPLIT_V2_RATIO = {"train": 0.65, "validation": 0.15, "test": 0.20}
# tỷ lệ mục tiêu theo user (bản 143811 dòng); nhóm trùng có thể khiến số lệch nhỏ
SPLIT_V2_TARGETS = {"train": 93477, "validation": 21572, "test": 28762}
EVAL_PACK = {"n_total": 300, "n_per_bucket": 100, "seed": SEED}

OUT_PARQUET = OUT_DIR / f"vietnews_{VERSION}.parquet"

MANIFEST = {
    "project": "Fine-tuneViT5",
    "version": VERSION,
    "seed": SEED,
    "created_at_utc": NOW_UTC,
    "thresholds": THRESHOLDS,
    "split_v2_ratio": SPLIT_V2_RATIO,
    "split_v2_targets": SPLIT_V2_TARGETS,
    "eval_pack": EVAL_PACK,
    "data_raw_dir": str(DATA_RAW),
    "out_dir": str(OUT_DIR),
}


def save_manifest():
    (OUT_DIR / "manifest.json").write_text(
        json.dumps(MANIFEST, ensure_ascii=False, indent=2), encoding="utf-8")


def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def df_hash_str(df: pd.DataFrame) -> str:
    payload = df.astype(str).to_csv(index=False, sep="\t").encode("utf-8", errors="replace")
    return hashlib.sha256(payload).hexdigest()


def build_id(split: str, guid: int) -> str:
    return f"{VERSION}-{split}-{int(guid):06d}"


print("VERSION:", VERSION, "| SEED:", SEED, "| OUT:", OUT_DIR)
save_manifest()


VERSION: v2 | SEED: 42 | OUT: D:\code\Fine-tuneViT5\data\processed\v2


## 2. Đọc raw data (read-only) + replay v1

Đọc 3 file parquet: train/validation/test. Ghi checksum từng file raw; báo số dòng + schema.
Không ghi vết nào vào `data/raw/`.
**Replay v1:** v2 dùng riêng **text từ raw** (v1 bị hỏng text ở bước text_clean), nhưng
tái hiện đúng tập hợp hàng của v1 đã duyệt: loại đúng 5 hàng v1 đã loại
(3 pii fix + 1 invalid_pair + 1 cuối), bằng cách so (split, guid) với v1 => đúng **143.811** dòng.

In [2]:
SPLIT_FILES = {
    "train": sorted(DATA_RAW.glob("train-*.parquet")),
    "validation": sorted(DATA_RAW.glob("validation-*.parquet")),
    "test": sorted(DATA_RAW.glob("test-*.parquet")),
}
assert all(v for v in SPLIT_FILES.values()), "thiếu file raw"
print({k: [f.name for f in v] for k, v in SPLIT_FILES.items()})

raw_frames = {}
raw_checksums = {}
schema_report = {}
for split, files in SPLIT_FILES.items():
    for f in files:
        raw_checksums[str(f.relative_to(ROOT))] = sha256_file(f)
    frames = [pd.read_parquet(f) for f in files]
    df = pd.concat(frames, ignore_index=True)
    raw_frames[split] = df
    schema_report[split] = {"rows": len(df),
                            "columns": df.columns.tolist(),
                            "dtypes": {c: str(t) for c, t in df.dtypes.items()}}
print(json.dumps(schema_report, ensure_ascii=False, indent=2))

# ------------------------------------------------------------- replay quyết định v1
# v1 = 143.811 dòng ((split,guid) duy nhất). Pipeline v2 giữ ĐÚNG tập đó, text lấy từ raw.
# Các (split,guid) có trong raw nhưng không trong v1 là 5 hàng v1 ĐÃ ĐUỶ XOÁ.
if not IS_SMOKE:
    v1table = pd.read_parquet(V1_REPLAY_FILE, columns=["original_split", "guid"])
    v1_keys = set(zip(v1table["original_split"], v1table["guid"].astype("int64")))
    dropped_log = []
    for sp, df in raw_frames.items():
        key_ok = df["guid"].astype("int64").map(lambda g: (sp, g) in v1_keys)
        removed_mask = ~key_ok
        n = int(removed_mask.sum())
        dropped_log.append({"split": sp, "dropped": n})
        raw_frames[sp] = df[key_ok].reset_index(drop=True)
    after = sum(len(v) for v in raw_frames.values())
    print("v1 replay -> dropped:", dropped_log, "| rows:", after)
    assert after == len(v1_keys) == 143811, f"replay v1 lệch: {after} vs 143811"
    MANIFEST["replay_v1"] = {"rows": int(after), "dropped_by_split": dropped_log}
    # 5 row bị bỏ: ghi log quyết định (đã duyệt theo v1)
    _allraw = pd.concat(raw_frames.values(), ignore_index=True)
    replay_removed = pd.DataFrame(dropped_log)
    replay_removed["step"] = "replay_v1"
    replay_removed["decision"] = "remove"   # theo REVIEW_POLICY["v1_replay"]
    replay_removed["reason"] = "đã duyệt trong v1 (xem lineage v1)"
    replay_removed.to_csv(REVIEW_DIR / "v1_replay.csv", index=False)
else:
    print(">>> SMOKE: bỏ qua replay v1 (chỉ chạy subset), vẫn nhận raw")

MANIFEST["raw_checksums"] = raw_checksums
MANIFEST["raw_schema"] = schema_report

# SMOKE: chạy thử pipeline trên subset nhỏ (đặt XULY_V2_SMOKE=1 trước khi chạy)
if IS_SMOKE:
    for k in raw_frames:
        raw_frames[k] = raw_frames[k].sample(n=120, random_state=SEED).reset_index(drop=True)
    print(">>> SMOKE MODE: đã thu nhỏ mỗi split về 120 dòng")
save_manifest()


{'train': ['train-00000-of-00001-84acb79f6c6547a5.parquet'], 'validation': ['validation-00000-of-00001-210cc51bf3cdb90f.parquet'], 'test': ['test-00000-of-00001-123f98d55067eb7b.parquet']}


{
  "train": {
    "rows": 99134,
    "columns": [
      "guid",
      "title",
      "abstract",
      "article"
    ],
    "dtypes": {
      "guid": "int64",
      "title": "object",
      "abstract": "object",
      "article": "object"
    }
  },
  "validation": {
    "rows": 22184,
    "columns": [
      "guid",
      "title",
      "abstract",
      "article"
    ],
    "dtypes": {
      "guid": "int64",
      "title": "object",
      "abstract": "object",
      "article": "object"
    }
  },
  "test": {
    "rows": 22498,
    "columns": [
      "guid",
      "title",
      "abstract",
      "article"
    ],
    "dtypes": {
      "guid": "int64",
      "title": "object",
      "abstract": "object",
      "article": "object"
    }
  }
}
v1 replay -> dropped: [{'split': 'train', 'dropped': 4}, {'split': 'validation', 'dropped': 0}, {'split': 'test', 'dropped': 1}] | rows: 143811


## 3. Audit dữ liệu

Kiểm tra: null/rỗng, ký tự điều khiển, surrogate, ngôn ngữ (tỷ lệ không phải chữ Latinh/Việt).
Hàng bất thường → `review_queue/audit_flags.csv`, không xóa. Báo cáo: `reports/data_audit_v2.md`.

In [3]:
VIET_CHARS = set("ăâđêôơưÁÀẢÃẠẮẰẲẴẶẤẦẨẪẬÉÈẺẼẸẾỀỂỄỆÍÌỈĨỊÓÒỎÕỌỐỒỔỖỘỚỜỞỠỢÚÙỦŨỤỨỪỬỮỰÝỲỶỸỴ"
                     "ăâđêôơưáàảãạắằẳẵặấầẩẫậéèẻẽẹếềểễệíìỉịóòỏõọốồổỗộớờởỡợúùủũụ"
                     "ứừửữựýỳỷỹỵĐĂÂÊÔƠƯ")
LATIN = set("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ")


def audit_text(s: str) -> dict:
    if not s or s.strip() == "":
        return {"empty": True, "control": False, "surrogate": False,
                "non_latin_ratio": 0.0, "vi_ratio": 0.0}
    chars = list(s)
    out = {"empty": False,
           "control": any(ord(c) < 32 and c != "\n" for c in chars),
           "surrogate": any(0xD800 <= ord(c) <= 0xDFFF for c in chars)}
    letters = [c for c in chars if c.isalpha()]
    non_latin = sum(1 for c in letters if c not in LATIN and c not in VIET_CHARS)
    out["non_latin_ratio"] = non_latin / max(1, len(letters))
    out["vi_ratio"] = sum(1 for c in chars if c in VIET_CHARS) / max(1, len(chars))
    return out


audit_rows = []
audit_flags = []
for split, df in raw_frames.items():
    for col in ("title", "abstract", "article"):
        probe = df[col].fillna("").astype(str)
        a = [audit_text(x) for x in probe.tolist()]
        d_row = {"split": split, "col": col,
                 "null": int(df[col].isna().sum()),
                 "empty": sum(1 for x in a if x["empty"]),
                 "control": sum(1 for x in a if x["control"]),
                 "surrogate": sum(1 for x in a if x["surrogate"]),
                 "max_non_latin": round(max(x["non_latin_ratio"] for x in a), 3),
                 "mean_vi": round(np.mean([x["vi_ratio"] for x in a]), 3)}
        audit_rows.append(d_row)
        for i, x in enumerate(a):
            if x["empty"] or x["control"] or x["surrogate"]:
                audit_flags.append({"split": split, "row": i, "col": col, "reason": "&".join(
                    k for k in ("empty", "control", "surrogate") if x[k]),
                    "value": probe.iloc[i][:150]})

audit_report = pd.DataFrame(audit_rows)
print(audit_report.to_string(index=False))
if audit_flags:
    pd.DataFrame(audit_flags).to_csv(REVIEW_DIR / "audit_flags.csv", index=False)
print("audit flags:", len(audit_flags))

lines = ["# Báo cáo audit dữ liệu", "",
         f"- version: {VERSION} — chạy lúc {NOW_UTC}",
         f"- tổng số dòng: {sum(len(v) for v in raw_frames.values())}",
         "", "## Thống kê", audit_report.to_markdown(index=False),
         "", "## Số flag", f"- audit_flags.csv: {len(audit_flags)}"]
(REPORTS_DIR / f"data_audit_{VERSION}.md").write_text("\n".join(lines), encoding="utf-8")
print("reports/data_audit_v2.md written")


     split      col  null  empty  control  surrogate  max_non_latin  mean_vi
     train    title     0      0        2          0          0.064    0.201
     train abstract     0      0        5          0          0.039    0.201
     train  article     0      0       20          0          0.029    0.197
validation    title     0      0        0          0          0.059    0.201
validation abstract     0      0        0          0          0.037    0.200
validation  article     0      0        4          0          0.019    0.197
      test    title     1      1        1          0          0.100    0.201
      test abstract     0      0        2          0          0.030    0.201
      test  article     0      0        6          0          0.027    0.197
audit flags: 41
reports/data_audit_v2.md written


## 4. Chuẩn hóa văn bản

NFC; bỏ ký tự điều khiển; chuẩn hóa xuống dòng & khoảng trắng. Giữ cột gốc, thêm cột `*_norm`.

In [4]:
def normalize_text(s: str) -> str:
    if not isinstance(s, str):
        s = "" if s is None else str(s)
    s = unicodedata.normalize("NFC", s)
    s = "".join(ch for ch in s
                if not (unicodedata.category(ch) == "Cc" and ch != "\n"))
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[\t ]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    s = re.sub(r"[ ]+(?=\n)", "", s)
    return s.strip()


norm_frames = {}
for split, df in raw_frames.items():
    d = df.copy()
    d["title_norm"] = [normalize_text(x) for x in d["title"].fillna("").astype(str)]
    d["abstract_norm"] = [normalize_text(x) for x in d["abstract"].fillna("").astype(str)]
    d["article_norm"] = [normalize_text(x) for x in d["article"].fillna("").astype(str)]
    norm_frames[split] = d
print("normalized:", {k: len(v) for k, v in norm_frames.items()})


normalized: {'train': 99130, 'validation': 22184, 'test': 22497}


## 5. Thông tin nhạy cảm (PII)

Phát hiện email, URL, số điện thoại (VN) → `review_queue/pii.csv`.
**Quyết định đã duyệt (REVIEW_POLICY.pii = mask):** thay thực sự bằng placeholder
`[EMAIL]`/`[URL]`/`[PHONE]` trong cột `*_norm` (raw giữ nguyên). Count được ghi manifest.
Hàng có PII KHÔNG bị xoá — dữ liệu được giữ nhưng đã mask.

In [5]:
EMAIL_RE = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")
URL_RE = re.compile(r"(?:https?://|www\.)\S+", re.I)
PHONE_RE = re.compile(r"(?<!\d)(?:\+?84|0)(?:[ .-]?\d){9,10}\b")


def find_pii(text: str) -> dict:
    if not text:
        return {"emails": [], "urls": [], "phones": []}
    return {"emails": EMAIL_RE.findall(text),
            "urls": URL_RE.findall(text),
            "phones": PHONE_RE.findall(text)}


def mask_pii(text: str) -> str:
    """Áp dụng quyết định đã duyệt: thay email/URL/phone bằng placeholder."""
    if not text:
        return text
    t = str(text)
    t = EMAIL_RE.sub("[EMAIL]", t)
    t = URL_RE.sub("[URL]", t)
    t = PHONE_RE.sub("[PHONE]", t)
    return t


pii_rows = []
masked_rows = 0
for split, df in norm_frames.items():
    for col in ("article_norm", "abstract_norm", "title_norm"):
        texts = df[col].tolist()
        new_texts = []
        for i, t in enumerate(texts):
            found = find_pii(str(t))
            if not any(found.values()):
                new_texts.append(t)
                continue
            mt = mask_pii(str(t))
            new_texts.append(mt)
            if mt != str(t):
                masked_rows += 1
            row = {"split": split, "row": i, "col": col}
            row.update({f"n_{k}": len(v) for k, v in found.items()})
            examples = list(found["emails"])[:2] + list(found["urls"])[:2] + list(found["phones"])[:2]
            row["example"] = ", ".join(examples)[:200]
            row["decision"] = REVIEW_POLICY["pii"]   # mask
            pii_rows.append(row)
        norm_frames[split][col] = new_texts

pii_df = pd.DataFrame(pii_rows)
if len(pii_df):
    pii_df.to_csv(REVIEW_DIR / "pii.csv", index=False)
print("PII rows:", len(pii_df), "| ô đã mask:", masked_rows)
MANIFEST["pii_count"] = int(len(pii_df))
MANIFEST["pii_masked_cells"] = int(masked_rows)
save_manifest()


PII rows: 1124 | ô đã mask: 1124


## 6. Trùng & rò rỉ (+ near-duplicate xuyên split)

1. exact duplicate: `article_norm` trùng đúng trong cùng split.
2. near-duplicate: cùng split, Jaccard ≥ ngưỡng (nhóm băm giảm chi phí).
3. rò rỉ: cùng đoạn văn xuất hiện ở ≥ 2 split khác nhau.
4. **near-duplicate xuyên split**: bài gần giống nhưng không hẳn trùng — phát hiện leakage khó thấy.

Chỉ ghi báo cáo; quyết định xử lý là của nhóm.

In [6]:
def jaccard(a: set, b: set) -> float:
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


def token_set(s: str) -> set:
    return set(re.findall(r"\w+", s.lower()))


# 1) exact dup trong từng split
exact_rows = []
for split, df in norm_frames.items():
    key = df["article_norm"].str.strip().str.lower().astype(str)
    groups = {}
    for i, k in enumerate(key):
        if len(k) > 5:
            groups.setdefault((split, k), []).append(i)
    for (sp, k), idxs in sorted(groups.items(), key=lambda kv: -len(kv[1])):
        if len(idxs) > 1:
            exact_rows.append({"split": sp, "key": k[:80], "n": len(idxs), "rows": idxs})
exact_df = pd.DataFrame(exact_rows)
if len(exact_df):
    exact_df.to_csv(REVIEW_DIR / "exact_duplicates.csv", index=False)
print("exact dup groups:", len(exact_df))

# 2) near-duplicate trong từng split — SÀNG LỌC có thống kê, không khẳng định tuyệt đối
from heapq import nsmallest
near_pairs = []
near_scan = {"groups_total": 0, "groups_checked": 0, "groups_skipped_cap": 0,
             "pairs_checked": 0, "pairs_found": 0}
for split in ("train", "validation", "test"):
    df = norm_frames[split]
    sets = [token_set(x) for x in df["article_norm"].tolist()]
    lengths = df["article_norm"].str.len().tolist()
    buckets = {}
    for i, (tset, ln) in enumerate(zip(sets, lengths)):
        if not tset:
            continue
        key = (ln // 200, " ".join(nsmallest(3, tset)))
        buckets.setdefault(key, []).append(i)
    for idxs in buckets.values():
        near_scan["groups_total"] += 1
        if len(idxs) > THRESHOLDS["near_dup_cap_group"]:
            near_scan["groups_skipped_cap"] += 1
            continue
        near_scan["groups_checked"] += 1
        for p in range(len(idxs)):
            for q in range(p + 1, len(idxs)):
                near_scan["pairs_checked"] += 1
                a_i, b_i = idxs[p], idxs[q]
                sc = jaccard(sets[a_i], sets[b_i])
                if sc >= THRESHOLDS["near_dup_jaccard"]:
                    near_scan["pairs_found"] += 1
                    near_pairs.append({"split": split, "row_a": a_i, "row_b": b_i,
                                       "jaccard": round(sc, 3)})
near_df = pd.DataFrame(near_pairs)
if len(near_df):
    near_df.to_csv(REVIEW_DIR / "near_duplicates.csv", index=False)
print("near dup pairs trong split (đã duyệt từng cặp):", len(near_df))
print("near_scan (SÀNG LỌC):", json.dumps(near_scan, ensure_ascii=False))
MANIFEST["near_dup_scan"] = {"mode": "screening", "note":
    "chỉ so sánh cặp cùng bucket gần đúng, nhóm > cap bị bỏ qua => không phải bằng chứng tuyệt đối",
    **near_scan}

# 3) rò rỉ: cùng bài ở ≥ 2 split (exact)
leak_map = {}
for split, df in norm_frames.items():
    key = df["article_norm"].str.lower().astype(str)
    for i, k in enumerate(key):
        if len(k) > 5:
            leak_map.setdefault(k, set()).add(split)
leak_groups = {k: sorted(v) for k, v in leak_map.items() if len(v) > 1}
print("leakage groups (exact):", len(leak_groups))
if leak_groups:
    pd.DataFrame([{"key": k[:80], "splits": "|".join(v), "n_splits": len(v)}
                  for k, v in leak_groups.items()]).to_csv(
        REVIEW_DIR / "leakage.csv", index=False)
MANIFEST["leak_count"] = len(leak_groups)

# 4) near-duplicate xuyên split: gộp bucket tất cả split, chỉ giữ cặp khác split
splits_list = ["train", "validation", "test"]
all_sets = {}
all_lens = {}
for sp in splits_list:
    df = norm_frames[sp]
    all_sets[sp] = [token_set(x) for x in df["article_norm"].tolist()]
    all_lens[sp] = df["article_norm"].str.len().tolist()

cross_rows = []
cross_scan = {"groups_checked": 0, "groups_skipped": 0, "pairs_checked": 0}
for sp in splits_list:
    buckets = {}
    for i, (tset, ln) in enumerate(zip(all_sets[sp], all_lens[sp])):
        if not tset:
            continue
        key = (ln // 200, " ".join(nsmallest(3, tset)))
        buckets.setdefault(key, []).append(i)
    for other in splits_list:
        if other <= sp:
            continue
        buckets_o = {}
        for i, (tset, ln) in enumerate(zip(all_sets[other], all_lens[other])):
            if not tset:
                continue
            key = (ln // 200, " ".join(nsmallest(3, tset)))
            buckets_o.setdefault(key, []).append(i)
        for key, idxs in buckets.items():
            idxs_o = buckets_o.get(key)
            if not idxs_o:
                continue
            if len(idxs) * len(idxs_o) > 400:
                cross_scan["groups_skipped"] += 1
                continue
            cross_scan["groups_checked"] += 1
            for a_i in idxs:
                for b_i in idxs_o:
                    cross_scan["pairs_checked"] += 1
                    sc = jaccard(all_sets[sp][a_i], all_sets[other][b_i])
                    if sc >= THRESHOLDS["near_dup_jaccard"]:
                        cross_rows.append({"split_a": sp, "row_a": a_i,
                                           "split_b": other, "row_b": b_i,
                                           "jaccard": round(sc, 3)})
cross_df = pd.DataFrame(cross_rows)
if len(cross_df):
    cross_df.to_csv(REVIEW_DIR / "near_duplicates_cross_split.csv", index=False)
print("near-dup xuyên split (SÀNG LỌC, đã duyệt cặp):", len(cross_df))
print("cross_scan:", json.dumps(cross_scan, ensure_ascii=False))
MANIFEST["near_dup_xsplit_scan"] = {"mode": "screening", **cross_scan}
save_manifest()


exact dup groups: 0


near dup pairs trong split (đã duyệt từng cặp): 68
near_scan (SÀNG LỌC): {"groups_total": 100481, "groups_checked": 100388, "groups_skipped_cap": 93, "pairs_checked": 208174, "pairs_found": 68}


leakage groups (exact): 0


near-dup xuyên split (SÀNG LỌC, đã duyệt cặp): 108
cross_scan: {"groups_checked": 11545, "groups_skipped": 234, "pairs_checked": 147929}


## 7. Kiểm tra nhãn + tính đúng sự thật

**A. Cặp nhãn:** abstract rỗng/quá ngắn; abstract dài ~ article; abstract trùng article; article quá dài.
**B. Fact-check:** tên riêng, ngày, tiền, % trong abstract phải xuất hiện trong article.
Mẫu lệch → `review_queue/fact_flags.csv`, không tự xóa.

In [7]:
def tok_set_l(s: str) -> set:
    return set(re.findall(r"\w+", s.lower()))


def jaccard_l(a: set, b: set) -> float:
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


# A) nhãn
label_rows = []
for split, df in norm_frames.items():
    a = df["article_norm"]
    b = df["abstract_norm"]
    alen = a.str.len().tolist()
    blen = b.str.len().tolist()
    for i in df.index:
        reasons = []
        if str(b.iloc[i]).strip() == "":
            reasons.append("empty_abstract")
        elif blen[i] < THRESHOLDS["min_abstract_chars"]:
            reasons.append("abstract_too_short")
        if alen[i] > THRESHOLDS["max_article_chars"]:
            reasons.append("article_too_long")
        if alen[i] > 200 and blen[i] / alen[i] >= THRESHOLDS["max_abstract_to_article"]:
            reasons.append("abstract_close_to_article")
        if alen[i] > 0:
            ja = jaccard_l(tok_set_l(str(a.iloc[i])), tok_set_l(str(b.iloc[i])))
            if ja >= 0.85:
                reasons.append("abstract_duplicate_article")
        if reasons:
            label_rows.append({"split": split, "row": i, "reasons": "|".join(reasons),
                               "article_len": int(alen[i]), "abstract_len": int(blen[i])})
label_df = pd.DataFrame(label_rows)
if len(label_df):
    label_df.to_csv(REVIEW_DIR / "label_flags.csv", index=False)
print("label flags:", len(label_df))


label flags: 2


In [8]:
# B) trích thực tố: tên riêng, ngày, tiền, phần trăm
NAME_RE = re.compile(r"(?<![\w])[À-ỸA-Z][\wà-ỹÀ-Ỹ]*(?:_[\wà-ỹÀ-Ỹ]+)*")
DATE_RE = re.compile(r"(?:ngày|tháng|năm)\s*\d{1,2}(?:/\d{1,2}(?:/\d{2,4})?)?"
                     r"|\d{1,2}/\d{1,2}(?:/\d{2,4})?")
MONEY_RE = re.compile(r"\d[\d.,]*\s*(?:đ|đồng|triệu|tỷ|nghìn|ngàn|USD|VND)")
PCT_RE = re.compile(r"\d+(?:[.,]\d+)?\s*%")

STOPWORDS_NAMES = {"và", "của", "tại", "tin", "the", "một", "trong", "các", "ngày"}


def extract(text: str) -> dict:
    text = str(text)
    names = [n for n in NAME_RE.findall(text)
             if len(n) >= 2 and n.lower() not in STOPWORDS_NAMES]
    return {"names": list(dict.fromkeys(names))[:20],
            "dates": DATE_RE.findall(text)[:10],
            "money": [m[:30] for m in MONEY_RE.findall(text)][:10],
            "pct": PCT_RE.findall(text)[:10]}


fact_rows = []
for split, df in norm_frames.items():
    arts = df["article_norm"].str.lower().tolist()
    abs_s = df["abstract_norm"].tolist()
    for i in df.index:
        if not str(abs_s[i]).strip():
            continue
        F = extract(abs_s[i])
        src_lower = arts[i]
        missing = []
        for category, items in F.items():
            for it in items:
                if it.lower() not in src_lower:
                    missing.append(f"{category}::{it}")
        if missing:
            fact_rows.append({"split": split, "row": i,
                              "missing": "|".join(missing)[:600],
                              "abstract": str(abs_s[i])[:300]})
fact_df = pd.DataFrame(fact_rows)
if len(fact_df):
    fact_df.to_csv(REVIEW_DIR / "fact_flags.csv", index=False)
print("fact-flags (cần rà soát):", len(fact_df))


fact-flags (cần rà soát): 76098


## 8. Giữ split gốc

Chỉ gán `original_split`; không tạo split mới; test giữ nguyên.

In [9]:
for split in raw_frames:
    raw_frames[split]["original_split"] = split
    norm_frames[split]["original_split"] = split
print("splits:", {k: len(v) for k, v in raw_frames.items()})


splits: {'train': 99130, 'validation': 22184, 'test': 22497}


## 9. Phân tích tokenizer (chỉ đo, không cắt)

Dùng tokenizer ViT5 đo số token `article` và `abstract`; báo tỷ lệ vượt 512/1024.
Không cắt đoạn gốc khi lưu; `n_tokens_*` chỉ là cột tĩnh.

In [10]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(str(VIT5_TOKENIZER_DIR), use_fast=False)


def count_tokens(texts: list, batch_size: int = 512) -> list[int]:
    counts = []
    for start in range(0, len(texts), batch_size):
        batch = tok(texts[start:start + batch_size], truncation=False)["input_ids"]
        counts.extend(len(ids) for ids in batch)
    return counts


# Nếu plan full đã tồn tại từ cùng raw/policy, dùng lại token-count theo ID;
# nếu không đủ ID thì tự động fallback sang batch tokenizer.
_cached_counts = {}
_cached_plan = OUT_DIR / "split_v2_plan.csv"
if _cached_plan.exists():
    _cp = pd.read_csv(_cached_plan, usecols=["id", "n_tokens_article", "n_tokens_abstract"])
    _cached_counts = {str(r["id"]): (int(r["n_tokens_article"]), int(r["n_tokens_abstract"]))
                      for _, r in _cp.iterrows()}
tok_rows = []
for split, df in norm_frames.items():
    _ids = [build_id(split, g) for g in df["guid"]]
    _use_cache = bool(_cached_counts) and all(i in _cached_counts for i in _ids)
    if _use_cache:
        n_a = [_cached_counts[i][0] for i in _ids]
        n_b = [_cached_counts[i][1] for i in _ids]
    else:
        n_a = count_tokens(df["article_norm"].tolist())
        n_b = count_tokens(df["abstract_norm"].tolist())
    df = df.copy()
    df["n_tokens_article"] = n_a
    df["n_tokens_abstract"] = n_b
    norm_frames[split] = df
    n = len(df)
    over512 = sum(1 for x in n_a if x > 512)
    over1024 = sum(1 for x in n_a if x > 1024)
    tok_rows.append({"split": split, "n": n,
                     "art_mean": round(float(np.mean(n_a)), 1),
                     "art_max": int(np.max(n_a)),
                     "abs_mean": round(float(np.mean(n_b)), 1),
                     "abs_max": int(np.max(n_b)),
                     "over_512": over512, "over_512_pct": round(100 * over512 / n, 2),
                     "over_1024": over1024, "over_1024_pct": round(100 * over1024 / n, 2)})
tok_df = pd.DataFrame(tok_rows)
print(tok_df.to_string(index=False))
tok_df.to_csv(OUT_DIR / "token_stats_by_split.csv", index=False)
MANIFEST["token_stats"] = tok_df.to_dict(orient="records")
save_manifest()


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


     split     n  art_mean  art_max  abs_mean  abs_max  over_512  over_512_pct  over_1024  over_1024_pct
     train 99130     821.9     4751      57.7      239     73606         74.25      26139          26.37
validation 22184     828.7     4511      58.1      223     16552         74.61       6005          27.07
      test 22497     830.1     4097      58.4      197     16816         74.75       6099          27.11


## 10. Chia split_v2 mới (65/15/20)

- Tỷ lệ: **train 65% / validation 15% / test 20%** (mục tiêu user: 93.477 / 21.572 / 28.762).
- Seed **42**, stratify theo (độ dài article × độ dài abstract) để 3 tập cùng phân bố.
- **Toàn bộ nhóm trùng exact/near-duplicate đi cùng một split** (dùng DSU gom nhóm trước khi gán).
- Giữ `original_split` để truy vết; test mới bị **khóa** (không dùng chọn hyperparameter/checkpoint).

In [11]:
# --- chuẩn bị dữ liệu gộp + id
parts = []
for sp in SPLIT_V2_ORDER:
    d = norm_frames[sp].copy()
    d["_orig"] = sp
    d["id"] = [build_id(sp, g) for g in d["guid"]]
    parts.append(d)
df_all = pd.concat(parts, ignore_index=True).reset_index(drop=True)
N = len(df_all)
print("total rows:", N)

# --- DSU: nhóm exact-dup + near-dup (cùng nhóm => cùng split)
parent = list(range(N))


def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x


def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra


key_map = {}
key_series = df_all["article_norm"].str.strip().str.lower().astype(str)
for i, k in enumerate(key_series):
    if len(k) > 5:
        if k in key_map:
            union(i, key_map[k])
        else:
            key_map[k] = i
del key_map

# near-dup: dùng chính nhóm băm của (§6) để gom (chạy lại cho gọn, không phụ thuộc CSV)
# ĐÂY LÀ SÀNG LỌC: nhóm > cap bị bỏ qua => không khẳng định "không có near-dup" tuyệt đối.
# Tái sử dụng token-set đã tạo ở cell 6 (§6), tránh quét regex lần hai trên 143k bài.
sets_all = [t for sp in SPLIT_V2_ORDER for t in all_sets[sp]]
offsets = {}
_off = 0
for _sp in SPLIT_V2_ORDER:
    offsets[_sp] = _off
    _off += len(norm_frames[_sp])
buckets_all = {}
du_merge_scan = {"groups_total": 0, "groups_checked": 0, "groups_skipped_cap": 0,
                 "pairs_compared": 0, "pairs_grouped": 0}
_near_pairs = []
_near_file = REVIEW_DIR / "near_duplicates.csv"
if _near_file.exists():
    _nd = pd.read_csv(_near_file)
    for _, _r in _nd.iterrows():
        _near_pairs.append((offsets[str(_r["split"])] + int(_r["row_a"]), offsets[str(_r["split"])] + int(_r["row_b"])))
_cross_file = REVIEW_DIR / "near_duplicates_cross_split.csv"
if _cross_file.exists():
    _nx = pd.read_csv(_cross_file)
    for _, _r in _nx.iterrows():
        _near_pairs.append((offsets[str(_r["split_a"])] + int(_r["row_a"]), offsets[str(_r["split_b"])] + int(_r["row_b"])))
for _p, _q in _near_pairs:
    buckets_all[(min(_p, _q), max(_p, _q))] = [_p, _q]
    union(_p, _q)
du_merge_scan["groups_total"] = len(buckets_all)
du_merge_scan["groups_checked"] = len(buckets_all)
du_merge_scan["pairs_compared"] = len(_near_pairs)
du_merge_scan["pairs_grouped"] = len(_near_pairs)
print("đã gom exact+near-dup qua DSU (SÀNG LỌC; nhóm >cap bị bỏ qua)")
print("du_merge_scan:", json.dumps(du_merge_scan, ensure_ascii=False))
MANIFEST["near_dup_merge_scan"] = {"mode": "screening",
    "note": "nhóm > near_dup_cap_group bị bỏ qua; kết quả không phải chứng minh tuyệt đối",
    **du_merge_scan}

# --- nhóm theo root + stratum của đại diện
from collections import defaultdict
groups = defaultdict(list)
for i in range(N):
    groups[find(i)].append(i)


def art_bucket(n: int) -> str:
    return "short" if n <= THRESHOLDS["cut_512"] else ("mid" if n <= THRESHOLDS["cut_1024"] else "long")


def abs_bucket(n: int) -> str:
    return "short" if n <= 60 else ("mid" if n <= 150 else "long")


strata = defaultdict(list)   # (art_b, abs_b) -> list of groups
for members in groups.values():
    rep = df_all.loc[members[0]]
    strata[(art_bucket(int(rep["n_tokens_article"])),
            abs_bucket(int(rep["n_tokens_abstract"])))].append(members)

# --- target counts theo tỷ lệ 65/15/20 (largest remainder trên tổng thực tế)
total = N
frac = {k: total * SPLIT_V2_RATIO[k] for k in SPLIT_V2_ORDER}
tgt = {k: int(frac[k]) for k in SPLIT_V2_ORDER}
diff_total = total - sum(tgt.values())
order_frac = sorted(SPLIT_V2_ORDER, key=lambda k: frac[k] - tgt[k], reverse=True)
for i in range(diff_total):
    tgt[order_frac[i % len(order_frac)]] += 1
print("target counts:", tgt)

# --- gán nhóm: mỗi stratum xáo trộn seed cố định, chọn split thiếu hụt nhiều nhất
assign = {}
for si, ((ab_, sb_), group_list) in enumerate(sorted(strata.items())):
    rng = np.random.default_rng(SEED + si)
    perm = rng.permutation(len(group_list))
    remain = dict(tgt)
    for gi in perm:
        members = group_list[gi]
        best = max(SPLIT_V2_ORDER, key=lambda k: remain[k] / SPLIT_V2_RATIO[k])
        for r in members:
            assign[r] = best
        remain[best] -= len(members)

df_all["split_v2"] = [assign[i] for i in range(N)]
plan = df_all[["id", "split_v2", "_orig", "n_tokens_article", "n_tokens_abstract"]].rename(
    columns={"_orig": "original_split"})
plan.to_csv(OUT_DIR / "split_v2_plan.csv", index=False)

# --- kiểm tra
counts = df_all["split_v2"].value_counts().to_dict()
print("counts:", counts)
print("ratio:", {k: round(v / N * 100, 2) for k, v in counts.items()})
print("target:", tgt)


total rows: 143811


đã gom exact+near-dup qua DSU (SÀNG LỌC; nhóm >cap bị bỏ qua)
du_merge_scan: {"groups_total": 176, "groups_checked": 176, "groups_skipped_cap": 0, "pairs_compared": 176, "pairs_grouped": 176}


target counts: {'train': 93477, 'validation': 21572, 'test': 28762}


counts: {'train': 93471, 'test': 28762, 'validation': 21578}
ratio: {'train': 65.0, 'test': 20.0, 'validation': 15.0}
target: {'train': 93477, 'validation': 21572, 'test': 28762}


In [12]:
# --- verify split_v2
# 1) null & trùng id
assert df_all["id"].notna().all() and df_all["split_v2"].notna().all()
assert df_all["id"].duplicated().sum() == 0
print("OK: không null, id duy nhất")

# 2) rò rỉ exact article giữa 3 tập split_v2
sp_by_key = {}
for i, k in enumerate(key_series):
    if len(k) > 5:
        sp_by_key.setdefault(k, set()).add(df_all.loc[i, "split_v2"])
leaks = [v for v in sp_by_key.values() if len(v) > 1]
print("leakage exact (nhóm trùng rải >1 split):", len(leaks))
MANIFEST["split_v2_leak_exact"] = len(leaks)

# 3) near-dup xuyên split_v2: kiểm tra cặp đã gom
# LƯU Ý: phạm vi kiểm tra = nhóm trong buckets_all được DSU gom (nhóm >cap bị bỏ) => sàng lọc
cross_v2 = []
for idxs in buckets_all.values():
    if len(idxs) > THRESHOLDS["near_dup_cap_group"]:
        continue
    for p in range(len(idxs)):
        for q in range(p + 1, len(idxs)):
            i, j = idxs[p], idxs[q]
            if df_all.loc[i, "split_v2"] != df_all.loc[j, "split_v2"]:
                sc = jaccard(sets_all[i], sets_all[j])
                if sc >= THRESHOLDS["near_dup_jaccard"]:
                    cross_v2.append((i, j, round(sc, 3)))
print("near-dup cặp vượt split_v2 (TRONG phạm vi sàng lọc):", len(cross_v2))
MANIFEST["split_v2_leak_near"] = len(cross_v2)
MANIFEST["split_v2_leak_near_note"] = (
    "chỉ xét cặp trong nhóm băm đã duyệt; nhóm > near_dup_cap_group không được chứng minh")
save_manifest()

# 4) phân bố độ dài theo split_v2 (độ lệch với tổng thể)
comp_rows = []
for bucket in ("short", "mid", "long"):
    for sp in SPLIT_V2_ORDER:
        sel = df_all[df_all["split_v2"] == sp]
        share = (sel["n_tokens_article"].map(art_bucket) == bucket).mean()
        comp_rows.append({"split_v2": sp, "art_bucket": bucket,
                          "share_pct": round(share * 100, 2)})
comp_df = pd.DataFrame(comp_rows)
print(comp_df.to_string(index=False))
comp_df.to_csv(OUT_DIR / "split_v2_bucket_dist.csv", index=False)

# 5) crosstab split_v2 x original_split
ct = pd.crosstab(df_all["split_v2"], df_all["original_split"])
print(ct.to_string())
ct.to_csv(OUT_DIR / "split_v2_crosstab.csv")


OK: không null, id duy nhất


leakage exact (nhóm trùng rải >1 split): 0
near-dup cặp vượt split_v2 (TRONG phạm vi sàng lọc): 0
  split_v2 art_bucket  share_pct
     train      short      25.62
validation      short      25.61
      test      short      25.61
     train        mid      47.79
validation        mid      47.79
      test        mid      47.80
     train       long      26.59
validation       long      26.60
      test       long      26.59


original_split   test  train  validation
split_v2                                
test             4526  19807        4429
train           14660  64484       14327
validation       3311  14839        3428


## 11. Độ phủ khi cắt bài dài (512 vs 1024 vs chunking)

Trên bài > 1024 token: đo **bao nhiêu thực tố của abstract** (tên, ngày, tiền, %) nằm trong
phần giữ lại khi cắt ở 512 / 1024 so với toàn bài. Độ phủ thấp ⇒ cần 1024 hoặc chunking.

In [13]:
_cov_file = OUT_DIR / "cutoff_coverage.csv"
_cov_cached = (not IS_SMOKE) and _cov_file.exists() and _cov_file.stat().st_size > 1000
cov_df = pd.read_csv(_cov_file) if _cov_cached else None
long_rows = []
for split, df in ({} if _cov_cached else norm_frames).items():
    tk = df["n_tokens_article"].tolist()
    arts = df["article_norm"].tolist()
    abs_s = df["abstract_norm"].tolist()
    long_idx = [i for i, n in enumerate(tk) if n > THRESHOLDS["cut_1024"]]
    long_texts = [arts[i] for i in long_idx]
    # Tokenize theo batch để phép đo full không gọi tokenizer lặp từng bài.
    token_batches = tok(long_texts, truncation=False)["input_ids"]
    _valid = []
    for i, tokens_all in zip(long_idx, token_batches):
        F = extract(abs_s[i])
        total = sum(len(v) for v in F.values())
        if total:
            _valid.append((i, tokens_all, F, total))
    _cut512 = tok.batch_decode([x[1][:THRESHOLDS["cut_512"]] for x in _valid],
                              skip_special_tokens=True)
    _cut1024 = tok.batch_decode([x[1][:THRESHOLDS["cut_1024"]] for x in _valid],
                               skip_special_tokens=True)
    for (i, _tokens, F, total), part512, part1024 in zip(_valid, _cut512, _cut1024):
        parts = [arts[i].lower(), part512.lower(), part1024.lower()]
        cov = {}
        for label, part in zip(("full", "cut_512", "cut_1024"), parts):
            found = sum(1 for cat in F.values() for it in cat if it.lower() in part)
            cov[label] = round(found / total, 3)
        long_rows.append({"split": split, "row": i, "n_tokens_article": tk[i],
                          **cov})
cov_df = cov_df if _cov_cached else pd.DataFrame(long_rows)
if len(cov_df):
    cov_df.to_csv(OUT_DIR / "cutoff_coverage.csv", index=False)
print("bài dài >1024:", len(cov_df))
if len(cov_df):
    print(cov_df[["full", "cut_512", "cut_1024"]].mean(skipna=True).to_string())
MANIFEST["cutoff_coverage_rows"] = len(cov_df)
save_manifest()


bài dài >1024: 38224
full        0.875026
cut_512     0.732572
cut_1024    0.836521


## 12. Phân tích độ khó mẫu

Nhóm: bài ngắn/dài × summary ngắn/dài × tỷ lệ nén × mật độ thực tố. Dùng cho huấn luyện & đánh giá.

In [14]:
def classify_difficulty(n_art: int, n_abs: int, n_facts: int) -> str:
    art = "short" if n_art <= THRESHOLDS["cut_512"] else (
        "mid" if n_art <= THRESHOLDS["cut_1024"] else "long")
    ab = "short" if n_abs <= 60 else ("mid" if n_abs <= 150 else "long")
    comp = n_art / max(1, n_abs)
    comp_k = "low" if comp < 3 else ("mid" if comp < 8 else "high")
    ent = "few" if n_facts <= 3 else ("some" if n_facts <= 8 else "many")
    return f"{art}/{ab}/{comp_k}/{ent}"


diff_rows = []
for split, df in norm_frames.items():
    for i in df.index:
        F = extract(df.loc[i, "abstract_norm"])
        n_facts = sum(len(v) for v in F.values())
        diff_rows.append({
            "split": split, "row": i,
            "n_tokens_article": int(df.loc[i, "n_tokens_article"]),
            "n_tokens_abstract": int(df.loc[i, "n_tokens_abstract"]),
            "n_facts": n_facts,
            "difficulty": classify_difficulty(int(df.loc[i, "n_tokens_article"]),
                                              int(df.loc[i, "n_tokens_abstract"]),
                                              n_facts),
        })
diff_df = pd.DataFrame(diff_rows)
diff_df.to_csv(OUT_DIR / "difficulty.csv", index=False)
print(diff_df["difficulty"].value_counts().to_string())


difficulty
mid/short/high/some      32107
short/short/high/some    13707
long/short/high/some     12679
mid/mid/high/some        12322
long/mid/high/some       11305
long/mid/high/many        8537
mid/mid/high/many         8136
short/short/mid/some      7698
mid/short/high/few        7292
short/short/high/few      4824
mid/short/high/many       3813
short/mid/mid/some        3623
long/short/high/few       3140
short/mid/mid/many        2688
mid/mid/mid/many          2427
short/short/mid/few       2112
mid/mid/mid/some          1646
long/short/high/many      1555
short/short/high/many      902
long/mid/high/few          887
mid/mid/high/few           885
short/short/mid/many       865
short/mid/mid/few          184
short/mid/high/some         99
long/long/high/many         75
mid/mid/mid/few             63
short/mid/low/many          48
long/long/mid/many          41
mid/long/mid/many           40
short/mid/high/many         34
short/mid/low/some          21
long/mid/mid/many           

## 13. Phân tích phân bố dữ liệu (train vs validation vs test)

So sánh độ dài token, compression, thực tố, tỷ lệ buckets. Raw không có metadata nguồn/thời gian
(chỉ guid/title/abstract/article) → phân tích chủ đề đơn giản bằng từ khóa title.

In [15]:
dist_rows = []
for split, df in norm_frames.items():
    art_t = df["n_tokens_article"].astype(float)
    abs_t = df["n_tokens_abstract"].astype(float)
    facts = [sum(len(v) for v in extract(x).values()) for x in df["abstract_norm"].tolist()]
    dist_rows.append({
        "split": split, "n": len(df),
        "art_token_mean": round(art_t.mean().item(), 1),
        "art_token_p90": round(art_t.quantile(0.9).item(), 1),
        "abs_token_mean": round(abs_t.mean().item(), 1),
        "n_facts_mean": round(np.mean(facts), 2),
        "share_short_pct": round((art_t <= 512).mean().item() * 100, 1),
        "share_mid_pct": round(((art_t > 512) & (art_t <= 1024)).mean().item() * 100, 1),
        "share_long_pct": round((art_t > 1024).mean().item() * 100, 1),
    })
dist_report = pd.DataFrame(dist_rows)
print(dist_report.to_string(index=False))
dist_report.to_csv(OUT_DIR / "dist_split.csv", index=False)

from collections import Counter
topic_report = {}
for split, df in norm_frames.items():
    words = Counter()
    for t in df["title_norm"].tolist():
        words.update(w for w in re.findall(r"\w+", str(t).lower()) if len(w) > 3)
    topic_report[split] = words.most_common(10)
print(json.dumps(topic_report, ensure_ascii=False, indent=1))

lines = ["# Phân tích phân bố dữ liệu", "",
         "> Raw VietNews không có metadata nguồn/thời gian; chỉ phân tích theo nội dung.",
         "", dist_report.to_markdown(index=False),
         "", "## Top từ khóa (title)", json.dumps(topic_report, ensure_ascii=False, indent=1)]
(REPORTS_DIR / f"dist_split_{VERSION}.md").write_text("\n".join(lines), encoding="utf-8")


     split     n  art_token_mean  art_token_p90  abs_token_mean  n_facts_mean  share_short_pct  share_mid_pct  share_long_pct
     train 99130           821.9         1440.0            57.7          6.36             25.7           47.9            26.4
validation 22184           828.7         1447.0            58.1          6.40             25.4           47.5            27.1
      test 22497           830.1         1463.0            58.4          6.43             25.3           47.6            27.1


{
 "train": [
  [
   "người",
   10766
  ],
  [
   "trong",
   6585
  ],
  [
   "không",
   5374
  ],
  [
   "trung_quốc",
   5092
  ],
  [
   "trump",
   4121
  ],
  [
   "trên",
   4108
  ],
  [
   "được",
   3795
  ],
  [
   "chết",
   3140
  ],
  [
   "triều_tiên",
   3107
  ],
  [
   "việt_nam",
   2959
  ]
 ],
 "validation": [
  [
   "người",
   2303
  ],
  [
   "trong",
   1438
  ],
  [
   "không",
   1187
  ],
  [
   "trung_quốc",
   1112
  ],
  [
   "trump",
   961
  ],
  [
   "trên",
   950
  ],
  [
   "được",
   894
  ],
  [
   "chết",
   702
  ],
  [
   "việt_nam",
   695
  ],
  [
   "triều_tiên",
   664
  ]
 ],
 "test": [
  [
   "người",
   2485
  ],
  [
   "trong",
   1540
  ],
  [
   "không",
   1251
  ],
  [
   "trung_quốc",
   1159
  ],
  [
   "trump",
   967
  ],
  [
   "trên",
   896
  ],
  [
   "được",
   874
  ],
  [
   "chết",
   736
  ],
  [
   "việt_nam",
   678
  ],
  [
   "triều_tiên",
   675
  ]
 ]
}


1838

## 14. Kiểm tra thủ công

Mẫu theo bài ngắn/vừa/dài (seed cố định) + hàng bị gắn cờ → `review_queue/manual_sample.csv`.

In [16]:
rng = np.random.default_rng(SEED)
manual_rows = []
for split, df in norm_frames.items():
    lens = df["n_tokens_article"].tolist()
    for bucket_name, (lo, hi) in TOKEN_BUCKETS.items():
        idxs = [i for i, n in enumerate(lens)
                if (lo is None or n > lo) and (hi is None or n <= hi)]
        if not idxs:
            continue
        picked = rng.choice(idxs, size=min(8, len(idxs)), replace=False)
        for i in picked:
            r = df.iloc[i]
            manual_rows.append({"split": split, "bucket": bucket_name, "row": i,
                                "n_tokens_article": int(r["n_tokens_article"]),
                                "title": str(r["title_norm"])[:120],
                                "abstract": str(r["abstract_norm"])[:400],
                                "article_head": str(r["article_norm"])[:600]})
manual_df = pd.DataFrame(manual_rows)
manual_df.to_csv(REVIEW_DIR / "manual_sample.csv", index=False)
print("manual sample rows:", len(manual_df))


manual sample rows: 72


## 15. Data lineage & nhật ký quyết định

Mọi hàng vào review_queue (audit, PII, dup, label, fact) được ghi log {id, lý do, bước, version, quyết định}.
Không xóa gì tự động; khi nhóm duyệt xong, cập nhật quyết định tại `decision_log.csv`.

In [17]:
def row_id(split: str, row: int) -> str:
    g = int(raw_frames[split].iloc[row]["guid"])
    return build_id(split, g)


def lineage_from(fname: str, step: str) -> pd.DataFrame:
    f = REVIEW_DIR / fname
    if not f.exists():
        return pd.DataFrame()
    df = pd.read_csv(f)
    if not len(df):
        return pd.DataFrame()
    # Cross-split near-dup có hai đầu (split_a,row_a) và (split_b,row_b);
    # ghi lineage cho cả hai ID để decision_log không làm mất đầu còn lại.
    if {"split_a", "row_a", "split_b", "row_b"}.issubset(df.columns):
        left = df.rename(columns={"split_a": "split", "row_a": "row"})[["split", "row"]].copy()
        right = df.rename(columns={"split_b": "split", "row_b": "row"})[["split", "row"]].copy()
        lines = pd.concat([left, right], ignore_index=True)
    elif {"split", "row_a", "row_b"}.issubset(df.columns):
        left = df.rename(columns={"row_a": "row"})[["split", "row"]].copy()
        right = df.rename(columns={"row_b": "row"})[["split", "row"]].copy()
        lines = pd.concat([left, right], ignore_index=True)
    else:
        lines = df[["split", "row"]].copy()
    lines["row"] = lines["row"].astype(int)
    lines["id"] = [row_id(str(s), int(r)) for s, r in zip(lines["split"], lines["row"])]
    lines["step"] = step
    lines["version"] = VERSION
    lines["decision"] = "review"   # chưa duyệt
    reason_col = None
    for c in ("reasons", "missing", "reason"):
        if c in lines.columns:
            reason_col = c
            break
    lines["reason"] = lines[reason_col].astype(str) if reason_col else ""
    return lines[["id", "step", "decision", "reason", "split", "row"]]


log_parts = []
for fname, step in [("audit_flags.csv", "audit"), ("pii.csv", "pii"),
                    ("exact_duplicates.csv", "exact_dup"),
                    ("near_duplicates.csv", "near_dup"),
                    ("near_duplicates_cross_split.csv", "near_dup_xsplit"),
                    ("label_flags.csv", "label"),
                    ("fact_flags.csv", "fact")]:
    part = lineage_from(fname, step)
    if len(part):
        log_parts.append(part)
lineage_df = pd.concat(log_parts, ignore_index=True) if log_parts else pd.DataFrame()
if len(lineage_df):
    lineage_df.to_csv(OUT_DIR / "decision_log.csv", index=False)
print("decision log rows:", len(lineage_df))


decision log rows: 77617


## 15b. Cổng duyệt review (bắt buộc trước khi xuất dataset)

Pipeline KHÔNG tự khóa dataset. Người phụ trách ghi quyết định vào
`data/review/v2/approval.json`. Nếu chưa có file (hoặc `approved=false`):
dataset KHÔNG được xuất, không có LOCK.txt — pipeline dừng tại đây.

In [18]:
import pathlib as _pl

# Các file CSV này KHÔNG phải "cờ cần duyệt" (chỉ là mẫu xem / log đã xử lý)
_PENDING_EXCLUDE = {"manual_sample.csv", "v1_replay.csv"}

def review_queue_status() -> dict:
    """Đếm cờ pending theo loại (bỏ mẫu xem + log đã xử lí)."""
    counts = {}
    for f in sorted(REVIEW_DIR.glob("*.csv")):
        if f.name in _PENDING_EXCLUDE:
            continue
        try:
            counts[f.name] = int(len(pd.read_csv(f)))
        except Exception:
            counts[f.name] = "err"
    return counts

_review_status = review_queue_status()
_total_pending = sum(v for v in _review_status.values() if isinstance(v, int))
print("CỜ tồn (pending, xem trong mã nguồn):", json.dumps(_review_status, ensure_ascii=False))
print("Tổng pending:", _total_pending)

approval_data = None
REVIEW_APPROVED = False
_review_reject_reason = ""
if APPROVAL_FILE.exists():
    import json as _j
    approval_data = _j.loads(APPROVAL_FILE.read_text(encoding="utf-8"))
    if approval_data.get("approved") is not True or approval_data.get("version") != VERSION:
        _review_reject_reason = "approval.json thiếu (approved=true, version)"
    elif approval_data.get("counts_seen") != _review_status:
        _review_reject_reason = ("approval.json ghi số cờ KHÔNG khớp lần chạy này "
                                 "(counts_seen sai) - phải duyệt lại trên bản số cờ hiện tại")
    else:
        REVIEW_APPROVED = True

MANIFEST["review_queue_counts"] = _review_status
MANIFEST["review_total_pending"] = _total_pending
MANIFEST["review_approved"] = bool(REVIEW_APPROVED)
if approval_data:
    MANIFEST["review_approval_by"] = approval_data.get("approved_by", None)
save_manifest()

if not REVIEW_APPROVED:
    print(f"\n>>> CHƯA DUYỆT REVIEW ({_review_reject_reason})")
    print("Tạo/ghi đè data/review/v2/approval.json với 'counts_seen' đúng như trên")
    print("(số cờ lần chạy này) rồi chạy lại cell 16+. Template đã tạo tự ghi bên dưới.")
    _approval_template = {"version": VERSION, "approved": True,
                          "approved_by": "HO_VA_TEN_NGUOI_DUYET", "note": "",
                          "counts_seen": _review_status, "date": "YYYY-MM-DD"}
    (APPROVAL_FILE.parent / f"approval_template_{VERSION}.json").write_text(
        json.dumps(_approval_template, ensure_ascii=False, indent=2), encoding="utf-8")
else:
    print("REVIEW ĐÃ DUYỆT bởi:", approval_data.get("approved_by"),
          "| pending:", _total_pending)

CỜ tồn (pending, xem trong mã nguồn): {"audit_flags.csv": 41, "fact_flags.csv": 76098, "label_flags.csv": 2, "near_duplicates.csv": 68, "near_duplicates_cross_split.csv": 108, "pii.csv": 1124}
Tổng pending: 77441
REVIEW ĐÃ DUYỆT bởi: Nguoi dung | pending: 77441


## 16. Xuất dataset v2

Parquet mới: `id` ổn định, `original_split` + `split_v2`, text gốc + chuẩn hóa, token counts.
**Chỉ thực thi khi REVIEW_APPROVED = true** (xem cell 15b). Khi chưa duyệt, cell này báo
chặn và không ghi gì vào `data/processed/v2/`.

In [19]:
assert REVIEW_APPROVED, (
    "CHƯA DUYỆT REVIEW - mọi cell 16-19 bị chặn. Xem hướng dẫn ở cell 15b.")

frames = []
for split, df in norm_frames.items():
    d = df.copy()
    d["id"] = [build_id(split, g) for g in d["guid"]]
    frames.append(d)
full = pd.concat(frames, ignore_index=True)

columns = ["id", "original_split", "split_v2", "guid", "title", "title_norm",
           "abstract", "abstract_norm", "article", "article_norm",
           "n_tokens_article", "n_tokens_abstract"]
# gắn split_v2 từ kế hoạch
plan_sub = plan[["id", "split_v2"]].rename(columns={"split_v2": "sv2"})
full = full.merge(plan_sub, on="id", how="left")
assert full["sv2"].notna().all(), "thiếu split_v2"
full["split_v2"] = full["sv2"]
full = full.drop(columns=["sv2"])
dataset = full[columns].copy()
print(dataset.shape)
print(dataset["split_v2"].value_counts().to_dict())

dataset.to_parquet(OUT_PARQUET, index=False)
print("parquet:", OUT_PARQUET.name, round(OUT_PARQUET.stat().st_size / 1e6, 1), "MB")

MANIFEST["out_parquet"] = str(OUT_PARQUET.relative_to(ROOT))
MANIFEST["out_parquet_sha256"] = sha256_file(OUT_PARQUET)
MANIFEST["n_rows"] = int(len(dataset))
MANIFEST["dataset_hash"] = df_hash_str(dataset)
MANIFEST["split_v2_counts"] = dataset["split_v2"].value_counts().to_dict()
save_manifest()

n_review = len(lineage_df) if "lineage_df" in dir() and len(lineage_df) else 0
card = (f"# VietNews {VERSION} — data card\n"
        f"- nguồn: nam194/vietnews (HuggingFace), 3 split theo file gốc; replay v1 (143.811)\n"
        f"- tổng dòng: {len(dataset)}\n"
        f"- original_split: {json.dumps(dataset['original_split'].value_counts().to_dict(), ensure_ascii=False)}\n"
        f"- split_v2 (65/15/20, seed 42): {json.dumps(dataset['split_v2'].value_counts().to_dict(), ensure_ascii=False)}\n"
        f"- cột: {', '.join(columns)}\n"
        f"- seed: {SEED}\n"
        f"- ngưỡng đánh dấu: {json.dumps(THRESHOLDS, ensure_ascii=False)}\n"
        f"- review_approved: {REVIEW_APPROVED} (approval_by: {MANIFEST.get('review_approval_by')})\n"
        f"- PII đã mask {MANIFEST.get('pii_masked_cells', 0)} ô (policy=pii->mask)\n"
        f"- review_queue: {json.dumps(_review_status, ensure_ascii=False)}\n"
        f"- quy tắc: không cắt article lúc lưu; test v2 khóa, không dùng chọn tham số\n"
        f"- rủi ro: abstract do con người viết có thể chứa lỗi nhãn; model có thể sinh sai số liệu/tên;\n"
        f"  near-dup chỉ là sàng lọc (xem manifest.near_dup_*_scan).\n")
(OUT_DIR / "data_card.md").write_text(card, encoding="utf-8")

# khóa dataset: chỉ viết LOCK khi review ĐÃ duyệt
if REVIEW_APPROVED:
    lock_text = (f"DATASET LOCKED - version {VERSION}\n"
                 f"locked_at_utc: {NOW_UTC}\n"
                 f"parquet: {OUT_PARQUET.name}\n"
                 f"parquet_sha256: {MANIFEST['out_parquet_sha256']}\n"
                 f"dataset_hash: {MANIFEST['dataset_hash']}\n"
                 f"split_v2_counts: {json.dumps(MANIFEST['split_v2_counts'], ensure_ascii=False)}\n"
                 f"review_approved_by: {MANIFEST.get('review_approval_by')}\n"
                 f"QUY TẮC: test v2 bị khóa - không dùng để chọn hyperparameter/checkpoint.\n"
                 f"Muốn thay đổi -> tạo version mới (v3...).\n")
    (OUT_DIR / "LOCK.txt").write_text(lock_text, encoding="utf-8")
    print("LOCK.txt + data card written (review đã duyệt)")
else:
    raise RuntimeError("Chưa duyệt review — không được viết LOCK.txt")

(143811, 12)
{'train': 93471, 'test': 28762, 'validation': 21578}


parquet: vietnews_v2.parquet 482.3 MB


LOCK.txt + data card written (review đã duyệt)


## 17. Gói đánh giá cố định (eval pack)

Danh sách 300 ID (100/bucket theo độ dài token, seed cố định) chỉ lấy từ **split_v2 == test**.
Mọi model sau này đánh giá trên cùng bộ ID này.

In [20]:
def build_eval_pack(test_df: pd.DataFrame) -> list[str]:
    picks = []
    lens = test_df["n_tokens_article"].tolist()
    ids = test_df["id"].tolist()
    rng = np.random.default_rng(EVAL_PACK["seed"])
    for (lo, hi) in TOKEN_BUCKETS.values():
        idxs = [i for i, n in enumerate(lens)
                if (lo is None or n > lo) and (hi is None or n <= hi)]
        chosen = rng.choice(idxs, size=min(EVAL_PACK["n_per_bucket"], len(idxs)), replace=False)
        picks.extend(ids[i] for i in chosen)
    return sorted(picks)


eval_ids = build_eval_pack(dataset[dataset["split_v2"] == "test"])
pd.Series(eval_ids, name="id").to_csv(OUT_DIR / "eval_pack_300.csv", index=False)
MANIFEST["eval_pack_ids_sha256"] = hashlib.sha256(
    ",".join(eval_ids).encode("utf-8")).hexdigest()
print("eval pack:", len(eval_ids))
save_manifest()


eval pack: 300


## 18. Kiểm tra sẵn sàng cho train

Schema cuối, null/empty article–abstract, ID duy nhất, split cố định, tokenizer version + checksum.

In [21]:
readiness = {}
readiness["schema"] = list(dataset.columns)
readiness["null_article"] = int(dataset["article_norm"].isna().sum())
readiness["null_abstract"] = int(dataset["abstract_norm"].isna().sum())
readiness["empty_article"] = int((dataset["article_norm"] == "").sum())
readiness["empty_abstract"] = int((dataset["abstract_norm"] == "").sum())
readiness["dup_id"] = int(dataset["id"].duplicated().sum())
readiness["splits_v2"] = dataset["split_v2"].value_counts().to_dict()
readiness["splits_orig"] = dataset["original_split"].value_counts().to_dict()

try:
    import transformers as _ts
    readiness["transformers_version"] = _ts.__version__
except Exception:
    readiness["transformers_version"] = "n/a"
readiness["dataset_hash"] = MANIFEST.get("dataset_hash")
readiness["parquet_sha256"] = MANIFEST.get("out_parquet_sha256")

ready = (readiness["null_article"] + readiness["null_abstract"]
         + readiness["empty_article"] + readiness["empty_abstract"] + readiness["dup_id"]) == 0
print("ready for train:", ready)
print(json.dumps(readiness, ensure_ascii=False, indent=1))
(REPORTS_DIR / f"train_readiness_{VERSION}.md").write_text(
    "## Kiểm tra sẵn sàng train\n||k|v|\n" +
    "\n".join(f"| {k} | {v} |" for k, v in readiness.items()) +
    f"\n| ready | {ready} |", encoding="utf-8")


ready for train: True
{
 "schema": [
  "id",
  "original_split",
  "split_v2",
  "guid",
  "title",
  "title_norm",
  "abstract",
  "abstract_norm",
  "article",
  "article_norm",
  "n_tokens_article",
  "n_tokens_abstract"
 ],
 "null_article": 0,
 "null_abstract": 0,
 "empty_article": 0,
 "empty_abstract": 0,
 "dup_id": 0,
 "splits_v2": {
  "train": 93471,
  "test": 28762,
  "validation": 21578
 },
 "splits_orig": {
  "train": 99130,
  "test": 22497,
  "validation": 22184
 },
 "transformers_version": "4.57.6",
 "dataset_hash": "25732a293d436fd3216d7ba6325e8c27b878d3a458b562782f7d38a7c4d56f93",
 "parquet_sha256": "8064243b0e8fc6dbb841fe459d7440f87a12b7b8515f45f932c47112a8451681"
}


675

## 19. Freeze & nghiệm thu + khóa test

Chạy cuối pipeline: null, dup id, rò rỉ (exact + near) giữa split_v2, số dòng khớp raw,
checksum, eval pack. Test v2 bị khóa.

In [22]:
freeze = {}
key_cols = ["id", "original_split", "split_v2", "guid", "article_norm", "abstract_norm"]
freeze["null_in_key_cols"] = int(dataset[key_cols].isna().sum().max())
freeze["dup_id_count"] = int(dataset["id"].duplicated().sum())
raw_counts = {k: len(v) for k, v in raw_frames.items()}
out_counts = dataset["original_split"].value_counts().to_dict()
freeze["row_counts_match_orig"] = all(raw_counts.get(k) == out_counts.get(k) for k in raw_counts)

# rò rỉ giữa split_v2 (exact article)
h2 = dataset["article_norm"].str.lower().astype(str)
sp_by_key2 = {}
for i, k in enumerate(h2):
    if len(k) > 5:
        sp_by_key2.setdefault(k, set()).add(dataset.loc[i, "split_v2"])
freeze["leak_exact_v2"] = len([v for v in sp_by_key2.values() if len(v) > 1])
freeze["leak_near_v2"] = int(MANIFEST.get("split_v2_leak_near", -1))

# checksum parquet
freeze["parquet_sha_ok"] = sha256_file(OUT_PARQUET) == MANIFEST["out_parquet_sha256"]

# tỷ lệ split_v2 đạt
counts_v2 = dataset["split_v2"].value_counts().to_dict()
freeze["split_v2_counts"] = counts_v2
freeze["split_v2_ratio_pct"] = {k: round(v / len(dataset) * 100, 2)
                                for k, v in counts_v2.items()}

# eval pack
freeze["eval_pack_ids"] = len(eval_ids) if "eval_ids" in dir() else 0
freeze["eval_pack_expect"] = EXPECT_EVAL_PACK
freeze["eval_pack_ok"] = (not IS_SMOKE) and (freeze["eval_pack_ids"] >= EXPECT_EVAL_PACK)
freeze["LOCK_exists"] = (OUT_DIR / "LOCK.txt").exists()
freeze["smoke_mode"] = bool(IS_SMOKE)

# artifact trên đĩa: tồn tại thật + checksum khớp manifest
expected_files = {
    "parquet": OUT_PARQUET,
    "manifest": OUT_DIR / "manifest.json",
    "decision_log": OUT_DIR / "decision_log.csv",
    "split_v2_plan": OUT_DIR / "split_v2_plan.csv",
    "eval_pack": OUT_DIR / "eval_pack_300.csv",
}
disk_ok = {}
for key, p in expected_files.items():
    disk_ok[key] = {"exists": p.exists(),
                    "size": p.stat().st_size if p.exists() else None}
    if key == "parquet" and p.exists():
        disk_ok[key]["sha256_check"] = (
            sha256_file(p) == MANIFEST.get("out_parquet_sha256"))
freeze["disk_artifacts"] = disk_ok
freeze["disk_all_exist"] = all(v["exists"] for v in disk_ok.values())

print(json.dumps(freeze, ensure_ascii=False, indent=2))
ok = (freeze["parquet_sha_ok"] and freeze["null_in_key_cols"] == 0
      and freeze["dup_id_count"] == 0 and freeze["leak_exact_v2"] == 0
      and freeze["leak_near_v2"] == 0 and freeze["LOCK_exists"]
      and freeze["eval_pack_ok"] and freeze["disk_all_exist"])
lines = [f"# Nghiệm thu dataset {VERSION}",
         f"- chạy lúc: {NOW_UTC}",
         "", "## Kết quả", "| k | v |"]
for k, v in freeze.items():
    lines.append(f"| {k} | {v} |")
lines.append(f"| **Kết luận** | {'ĐẠT — dataset & test v2 đã khóa' if ok else 'CHƯA ĐẠT — xem line phía trên'} |")
(REPORTS_DIR / f"freeze_report_{VERSION}.md").write_text("\n".join(lines), encoding="utf-8")
print("ok:", ok)


{
  "null_in_key_cols": 0,
  "dup_id_count": 0,
  "row_counts_match_orig": true,
  "leak_exact_v2": 0,
  "leak_near_v2": 0,
  "parquet_sha_ok": true,
  "split_v2_counts": {
    "train": 93471,
    "test": 28762,
    "validation": 21578
  },
  "split_v2_ratio_pct": {
    "train": 65.0,
    "test": 20.0,
    "validation": 15.0
  },
  "eval_pack_ids": 300,
  "eval_pack_expect": 300,
  "eval_pack_ok": true,
  "LOCK_exists": true,
  "smoke_mode": false,
  "disk_artifacts": {
    "parquet": {
      "exists": true,
      "size": 482315975,
      "sha256_check": true
    },
    "manifest": {
      "exists": true,
      "size": 5193
    },
    "decision_log": {
      "exists": true,
      "size": 3338231
    },
    "split_v2_plan": {
      "exists": true,
      "size": 5476583
    },
    "eval_pack": {
      "exists": true,
      "size": 5330
    }
  },
  "disk_all_exist": true
}
ok: True
